# 02 — Data Exploration

EDA across all datasets. Run `01_data_acquisition.ipynb` first.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, "../src")

from autobahn_safety.data_loaders import load_destatis_autobahn, load_unfallatlas
from autobahn_safety.preprocessing import clean_unfallatlas

DATA_RAW = Path("../data/raw")
sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## 1. Unfallatlas (Germany) — GPS accident records 2016–2024

In [ ]:
df_ua = load_unfallatlas(DATA_RAW / "germany" / "unfallatlas", list(range(2016, 2025)))
df_ua = clean_unfallatlas(df_ua)
print(f"Shape: {df_ua.shape}")
print(f"Columns: {list(df_ua.columns)}")
df_ua.head(3)

In [ ]:
# Records per year
by_year = df_ua.groupby("year").size().reset_index(name="count")
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_year["year"], by_year["count"], color="steelblue")
ax.set_title("Unfallatlas: accident records per year")
ax.set_xlabel("Year")
ax.set_ylabel("Records")
plt.tight_layout()
plt.show()
by_year

In [ ]:
# Severity breakdown
sev = df_ua.groupby(["year", "severity"]).size().unstack(fill_value=0)
sev.plot(kind="bar", stacked=True, figsize=(11, 5), colormap="RdYlGn_r")
plt.title("Accident severity by year (Unfallatlas)")
plt.ylabel("Count")
plt.xlabel("Year")
plt.tight_layout()
plt.show()

In [ ]:
# Missing values
missing = df_ua.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0].to_string())

In [ ]:
import math

# Spatial distribution (Germany bounding box)
sample = df_ua.sample(min(50_000, len(df_ua)), random_state=42).dropna(subset=["lon", "lat"])
fig, ax = plt.subplots(figsize=(8, 10))
sc = ax.scatter(sample["lon"], sample["lat"], c="steelblue", s=0.3, alpha=0.3)
ax.set_title("Accident locations (50k sample)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect(1 / math.cos(math.radians(51)))
plt.tight_layout()
plt.show()

## 2. Destatis Autobahn time series (1979–2021)

In [ ]:
df_de = load_destatis_autobahn(
    DATA_RAW / "germany" / "destatis" / "destatis_verkehrsunfaelle_zeitreihen.xlsx"
)
print(df_de.dtypes)
df_de.tail(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(df_de["year"], df_de["accidents_personal_injury"], marker="o", ms=3, color="steelblue")
axes[0].set_title("Autobahn: accidents with personal injury")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Accidents")

axes[1].plot(df_de["year"], df_de["accidents_fatal"], marker="o", ms=3, color="crimson")
axes[1].set_title("Autobahn: fatal accidents")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Fatal accidents")

plt.suptitle("German Autobahn accident trends (Destatis)", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Netherlands — RWS accidents

In [ ]:
rws_files = sorted((DATA_RAW / "netherlands" / "rws").glob("rws_accidents_*.csv"))
print(f'Available years: {[f.stem.split("_")[-1] for f in rws_files]}')

# Load one year for EDA
df_rws = pd.concat([pd.read_csv(f) for f in rws_files], ignore_index=True)
df_rws["year"] = df_rws["JAAR_VKL"]
print(f"Total records: {len(df_rws):,}")
df_rws.head(3)

In [ ]:
# Speed limit distribution — key for motorway identification
top_speeds = df_rws["MAXSNELHD"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(9, 4))
top_speeds.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("NL accidents by posted speed limit")
ax.set_xlabel("Speed limit (km/h)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()
print("Motorway (100-130 km/h):", df_rws[df_rws["MAXSNELHD"].isin([100, 120, 130])].shape[0])

In [ ]:
# Severity
print("AP3_CODE (severity):")
print(df_rws["AP3_CODE"].value_counts().to_string())

## 4. Summary statistics

In [ ]:
print("=== Germany Unfallatlas ===")
print(f"  Records   : {len(df_ua):,}")
print(f'  Years     : {df_ua["year"].min()} – {df_ua["year"].max()}')
print(f'  Fatal     : {(df_ua["severity"] == "fatal").sum():,}')
print(f'  Serious   : {(df_ua["severity"] == "serious_injury").sum():,}')
print(f'  Slight    : {(df_ua["severity"] == "slight_injury").sum():,}')
print()
print("=== Destatis Autobahn ===")
print(f'  Years     : {df_de["year"].min()} – {df_de["year"].max()}')
print(f"  Last year : {df_de.iloc[-1].to_dict()}")
print()
if "df_rws" in dir():
    print("=== RWS Netherlands ===")
    print(f"  Records   : {len(df_rws):,}")
    mway = df_rws[df_rws["MAXSNELHD"].isin([100, 120, 130])]
    print(f"  Motorway  : {len(mway):,} ({len(mway)/len(df_rws)*100:.1f}%)")